# Variables, Naming Conventions, and Dynamic Typing

**Topic:** `1.4` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Validate a snake_case name

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, naming, validation

**Explanation and reasoning**

isidentifier() enforces the character rules and rejects a leading digit, while the lower-case comparison rejects CapWords and mixed case. Rejecting keywords is a bonus check: a name that is a legal identifier can still be unusable as a variable.

In [ ]:
import keyword


def safe_name(name: str) -> bool:
    if not isinstance(name, str) or not name:
        return False
    if keyword.iskeyword(name):
        return False
    if not name.isidentifier():
        return False
    return name == name.lower()

**Complexity**

Time O(n) in the name length; space O(1).

**Edge cases and failure modes**

'_private' passes because it is a legal identifier that is already lower case.

**Alternative approaches**

A regular expression expresses the same rule in one line.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert safe_name('battery_pct_2') is True
assert safe_name('BatteryPct') is False
assert safe_name('2fast') is False
assert safe_name('class') is False

---

## Solution 2 — Describe a binding

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** type(), repr(), id()

**Explanation and reasoning**

Catching TypeError from len() is the portable way to ask 'does this object have a length' without enumerating the types that do. The type name comes from the class rather than a hard-coded mapping, so new types are handled automatically.

In [ ]:
def describe_binding(value) -> dict:
    try:
        length = len(value)
    except TypeError:
        length = None
    return {
        'type': type(value).__name__,
        'repr': repr(value),
        'length': length,
    }

**Complexity**

Time O(n) for repr on a large object; space O(n).

**Edge cases and failure modes**

An object whose __len__ raises ValueError would propagate; that is deliberate.

**Alternative approaches**

A match statement on type(value) is explicit but needs a branch per type.

**Tests — run the cell to verify the reference solution**

In [ ]:
d = describe_binding(48.0)
assert d['type'] == 'float'
assert d['length'] is None
assert describe_binding([1, 2])['length'] == 2

---

## Solution 3 — Swap two values with unpacking

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** tuples, unpacking, assignment

**Explanation and reasoning**

The parentheses turn the two simultaneous assignments into a tuple that can be returned. Both right-hand sides are evaluated before either name is rebound, which is what makes the swap correct without a temporary.

In [ ]:
def swap(a, b):
    return (b, a)

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

Equal values return a two-tuple unchanged, as expected.

**Alternative approaches**

temp = a; a = b; b = temp works but is three lines and needs a third name.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert swap('front', 'rear') == ('rear', 'front')
assert swap(1, 1) == (1, 1)
assert swap(1, 'x') == ('x', 1)

---

## Solution 4 — Parse an assignment line

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, split, validation

**Explanation and reasoning**

partition splits on the first '=' only, so a value that itself contains an equals sign survives intact. Stripping both halves removes the whitespace around the operator, and validating the name before returning stops a malformed target from reaching the caller.

In [ ]:
def parse_assignment(line: str) -> tuple[str, str]:
    if not isinstance(line, str) or not line.strip():
        raise ValueError('empty assignment line')
    name, separator, value = line.partition('=')
    if not separator:
        raise ValueError(f'no assignment in {line!r}')
    name, value = name.strip(), value.strip()
    if not name.isidentifier():
        raise ValueError(f'invalid name: {name!r}')
    return name, value

**Complexity**

Time O(n) in the line length; space O(n).

**Edge cases and failure modes**

A line with no '=' raises ValueError; an empty name raises ValueError.

**Alternative approaches**

str.split('=', 1) is equivalent but discards the separator flag.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert parse_assignment('battery_pct = 48.0') == ('battery_pct', '48.0')
assert parse_assignment('a = 1 = 2') == ('a', '1 = 2')

---

## Solution 5 — Convert a label to snake_case

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, case conversion, loops

**Explanation and reasoning**

Non-alphanumeric characters become spaces first, so separators are handled uniformly. A separator is inserted before an upper-case letter when the previous character is lower-case or a digit, or when we are leaving an acronym - that second rule is what turns 'SpeedMPS' into 'speed_mps' instead of 'speedmps'. Filtering empty parts collapses repeated separators.

In [ ]:
def to_snake(text: str) -> str:
    chars = [c if c.isalnum() else ' ' for c in text]
    out: list[str] = []
    for index, ch in enumerate(chars):
        if ch == ' ':
            out.append('_')
            continue
        if ch.isupper():
            prev = chars[index - 1] if index else ''
            nxt = chars[index + 1] if index + 1 < len(chars) else ''
            boundary = prev.islower() or prev.isdigit()
            acronym = prev.isupper() and nxt.islower()
            if prev and prev != ' ' and (boundary or acronym):
                out.append('_')
        out.append(ch.lower())
    parts = [p for p in ''.join(out).split('_') if p]
    return '_'.join(parts)

**Complexity**

Time O(n) in the text length; space O(n).

**Edge cases and failure modes**

Empty input returns an empty string; already-snake input is unchanged.

**Alternative approaches**

Two regular-expression passes are shorter but less explicit about the acronym rule.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert to_snake('MaxSpeedMPS') == 'max_speed_mps'
assert to_snake('max speed mps') == 'max_speed_mps'
assert to_snake('battery_pct') == 'battery_pct'

---

## Solution 6 — Find duplicate names with their positions

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, sets, enumeration

**Explanation and reasoning**

setdefault creates the empty list on first sight and appends thereafter, so one pass collects every position. Filtering to lists longer than one keeps unique names out of the result, and sorting makes the output stable.

In [ ]:
def find_duplicates(names) -> dict:
    positions: dict = {}
    for index, name in enumerate(names, start=1):
        positions.setdefault(name, []).append(index)
    return {n: sorted(p) for n, p in positions.items() if len(p) > 1}

**Complexity**

Time O(n log n) in the worst case from sorting; space O(n).

**Edge cases and failure modes**

An empty input returns an empty dict; all-identical input yields one key with every position.

**Alternative approaches**

collections.Counter plus a second pass is shorter and slower.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert find_duplicates(['a', 'b', 'a', 'c', 'a']) == {'a': [1, 3, 5]}
assert find_duplicates([]) == {}
assert find_duplicates(['a', 'b']) == {}

---

## Solution 7 — Make an independent copy of a sequence

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** copying, lists, mutability

**Explanation and reasoning**

list() always allocates a new container, so appending or removing at the top level cannot affect the caller's data. The docstring is explicit that this is shallow, which is the detail most often missed when a nested list appears to mutate unexpectedly.

In [ ]:
def clone(sequence):
    """Return a shallow copy that is safe to mutate at the top level.

    Nested mutable objects are shared with the original; use
    copy.deepcopy when independent nested state is required.
    """
    return list(sequence)

**Complexity**

Time O(n); space O(n).

**Edge cases and failure modes**

An empty input produces a new list rather than the same object.

**Alternative approaches**

sequence[:] and sequence.copy() are equivalent for list inputs.

**Tests — run the cell to verify the reference solution**

In [ ]:
src = [1, 2]
copy = clone(src)
copy.append(3)
assert src == [1, 2]
assert copy == [1, 2, 3]

---

## Solution 8 — List the public names in a namespace

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** globals, dicts, filtering

**Explanation and reasoning**

The filter expresses the three independent exclusions: private by convention, not a valid identifier, and not an imported module. Sorting gives a deterministic result, which matters when the list is compared against an expected value in a test.

In [ ]:
import types


def public_names(namespace: dict) -> list[str]:
    return sorted(
        name
        for name, value in namespace.items()
        if not name.startswith('_')
        and name.isidentifier()
        and not isinstance(value, types.ModuleType)
    )

**Complexity**

Time O(n log n); space O(n).

**Edge cases and failure modes**

An empty namespace returns an empty list without raising.

**Alternative approaches**

vars(module) is often a more convenient input than globals().

**Tests — run the cell to verify the reference solution**

In [ ]:
assert public_names({'_hidden': 1, 'robot_id': 'r1', 'MAX_SPEED': 1.5}) == ['MAX_SPEED', 'robot_id']
assert public_names({}) == []

---

## Solution 9 — Build a validated configuration

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, validation, error reporting

**Explanation and reasoning**

Copying the defaults first means neither argument is mutated, and update() applies overrides before any validation so every check sees the final value. On a failed cast the default is restored and a message is appended, which keeps the function total: it reports every problem rather than stopping at the first.

In [ ]:
def build_config(raw: dict, defaults: dict, casts: dict) -> tuple[dict, list[str]]:
    config = dict(defaults)
    config.update(raw)
    errors: list[str] = []
    for key in sorted(config):
        cast = casts.get(key)
        if cast is None:
            errors.append(f'unknown setting: {key}')
            continue
        try:
            config[key] = cast(config[key])
        except (TypeError, ValueError):
            config[key] = defaults.get(key)
            errors.append(f'cannot convert {key}')
    return config, sorted(errors)

**Complexity**

Time O(k log k) for k keys; space O(k).

**Edge cases and failure modes**

An unknown key adds an error and leaves the value untouched.

**Alternative approaches**

A dataclass with __post_init__ validation enforces this at construction time.

**Tests — run the cell to verify the reference solution**

In [ ]:
cfg, errs = build_config({'robot_id': 'r2'}, {'robot_id': 'r1', 'battery_wh': 48.0}, {'robot_id': str, 'battery_wh': float})
assert cfg == {'robot_id': 'r2', 'battery_wh': 48.0}
assert errs == []

---

## Solution 10 — Apply symbol renames without collisions

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dicts, ordering, strings

**Explanation and reasoning**

Sorting keys by descending length is the whole trick: replacing the longer name first stops a short alias from corrupting it. Word boundaries ensure only whole identifiers are touched, and re.escape protects names that contain regex metacharacters.

In [ ]:
import re


def rename_symbol(mapping: dict, text: str) -> str:
    out = text
    for old in sorted(mapping, key=len, reverse=True):
        pattern = rf'\b{re.escape(old)}\b'
        out = re.sub(pattern, mapping[old], out)
    return out

**Complexity**

Time O(k * n) for k keys over n characters; space O(n).

**Edge cases and failure modes**

Text with no matches is returned unchanged; punctuation around words is preserved.

**Alternative approaches**

An ast-based rename is safer for real refactoring but cannot rewrite comments or strings.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert rename_symbol({'b': 'battery', 'battery': 'level'}, 'b and battery') == 'battery and level'
assert rename_symbol({'x': 'y'}, 'no match') == 'no match'

---

# Robotics challenge solution — ROBO-X Challenge: Telemetry Settings Gate

**Explanation**

dict(DEFAULTS) protects the shipped defaults from being overwritten, and the comprehension applies every declared cast in one place so the values leaving this function are already the right type. safe_readings isolates each channel so a single faulty sensor degrades one entry rather than the whole report.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot, SensorError  # noqa: E402

DEFAULTS = {'battery_wh': 48.0, 'name': 'rover-01'}
CASTS = {'battery_wh': float, 'name': str}


def telemetry_config(robot) -> dict:
    """Merge the robot's own state over the shipped defaults."""
    state = robot.status()
    raw = {'battery_wh': state['battery_wh'], 'name': state['name']}
    config = dict(DEFAULTS)
    config.update(raw)
    return {key: CASTS[key](value) for key, value in config.items()}


def safe_readings(robot, channels) -> dict:
    readings = {}
    for channel in channels:
        try:
            readings[channel] = robot.read_sensor(channel)
        except SensorError:
            readings[channel] = None
    return readings

**Complexity**

Time O(k + c) for k settings and c channels; space O(k + c).

**Notes and trade-offs**

An unknown channel records None rather than raising.

In [ ]:
robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
cfg = telemetry_config(robot)
assert cfg['name'] == 'rover-01'
assert isinstance(cfg['battery_wh'], float)
assert safe_readings(robot, ['nope'])['nope'] is None

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.